In [2]:
import os
os.environ["USER_AGENT"] = "langchain-rag-app"

from langchain_huggingface import HuggingFaceEmbeddings
from typing import Annotated, Literal, Sequence, TypedDict
from langchainhub import Client
from pydantic import BaseModel, Field  # Import from pydantic directly

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create hub client for pulling prompts
hub = Client()

In [34]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3254.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")

In [36]:
messages = llm.invoke("hii how are you?")
print(messages.content)

Hello, I'm just a computer program, so I don't have feelings, but I'm functioning properly and ready to help you with any questions or tasks you have. How can I assist you today?


In [17]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

In [18]:
docs=[WebBaseLoader(url).load() for url in urls ]

In [19]:
docs

[[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final res

In [20]:
docs[0][0].metadata

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'title': "LLM Powered Autonomous Agents | Lil'Log",
 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\

In [21]:
docs_split=[items for sublist in docs for items in sublist]


In [22]:
text_split=RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=100, chunk_overlap=5)

In [23]:
docs_splitter=text_split.split_documents(docs_split)


In [25]:
vector_store=Chroma.from_documents(
    documents=docs_splitter,
    collection_name="langraph-rag",
    embedding=embeddings
)

In [26]:
retriever = vector_store.as_retriever()

In [27]:
from langchain_core.tools import create_retriever_tool


retreiver_tool=create_retriever_tool(retriever
                                     , name="retriever_tool"
    , description="useful for when you want to answer questions about the content of the urls"
)

In [28]:
tools=[retreiver_tool]

In [29]:
def ai_assistant(state):
    messages = state["message"]
    model_with_tool=llm.bind_tools(tools)

    response = model_with_tool.invoke(messages)
    return {"message": [response]}


    

In [30]:
class AgentState(TypedDict):
    message: Annotated[Sequence[BaseMessage], add_messages]

In [31]:
class grade(BaseModel):
       binary_score:str=Field(description="Relevance score 'yes' or 'no'")

In [33]:
def grade_document(state:AgentState):
    llm_with_structured_output = llm.with_structured_output(grade)
    prompt=PromptTemplate(
        template="You are a helpful assistant for grading the relevance of retrieved documents. Given the user query and the retrieved document, please provide a relevance score of 'yes' or 'no'.\n\nUser Query: {question}\nRetrieved Document: {context}\n\nRelevance Score (yes/no):",
        input_variables=["question","context"]
    )